# Model Creation & Analysis

Two models will be created with Meta's Prophet. The first one will be a baseline with just the rental data while the other will use the unemployment of VA as an additional regressor. Afterwards, the models will be finetuned and analysis will be performed to examine the performances.

## Outline 
* Base model creation
* Model with additional unemployment regressor
* Midway comparison
* Fine tuning
* Statistical Analysis

In [1]:
from pyprojroot.here import here
import pandas as pd
import os

# Set the cwd to the project's root
os.chdir(here())
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/jam/Documents/ds/MI1-2


In [2]:
import pandas as pd
# Create dataframe for combined data
combined_data = pd.read_csv('DATA/Charlottesville_Rent_Employment_Master.csv')

# Preview data
print( "COMBINED DATA")
combined_data.head(10)

COMBINED DATA


,ds,y,employment_count
0,2015-01-31,1030.293753,103503.0
1,2015-02-28,1033.140296,105522.0
2,2015-03-31,1036.707494,105700.0
3,2015-04-30,1057.235869,107064.0
4,2015-05-31,1060.458311,105949.0
5,2015-06-30,1066.297180,104874.0
6,2015-07-31,1063.034752,104150.0
7,2015-08-31,1071.748772,102822.0
8,2015-09-30,1080.462792,105301.0
9,2015-10-31,1102.797649,105007.0


First we have to account for COVID. Prophet has a "holiday" feature which we will use to account for the dip in our data. Through looking at our graphs and reflecting on our experience we decided on Jan - March 2020 with a lower window of 1 month and upper window of 3 months.

* `lower_window` : effect of the holiday that extends *before* the holiday begins
* `upper_window` : effect of the holiday that extends *after* the holiday ends

In [3]:
covid_dates = pd.date_range(start='2020-01-01', end='2020-03-31')
covid = pd.DataFrame({
  'holiday': 'covid',
  'ds': covid_dates,
  'lower_window': -30, # Lower window = effects that may have occured prior to covid ~ 1 month
  'upper_window': 90, # Upper window = effects that may have lasted past covid ~ 3 months
})


## Model 1: Base Model Creation

No NAN values should exist as the data was interpolated previously. Fitting the model will be limited to dates in 2024 so we can compare with the actual values in 2025.

In [4]:
from prophet import Prophet

# Take only the date and rental data from the combined data
rental_data = combined_data[['ds', 'y']].copy()
# Limit to 2024
rental_data = rental_data[rental_data['ds'] <= '2024-12-31']

m1 = Prophet(holidays=covid)
m1.fit(rental_data)

# Build the model excluding the year of 2025 
future = m1.make_future_dataframe(periods=12, freq='ME')
# Predict 2025 data
forecast1 = m1.predict(future)

# Create a data frame with yhat as the predicted value per row and columns for uncertainty levels
forecast1[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

Importing plotly failed. Interactive plots will not work.
16:15:23 - cmdstanpy - INFO - Chain [1] start processing
16:15:23 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: Run

,ds,yhat,yhat_lower,yhat_upper
120,2025-01-31,1831.227179,1812.205556,1851.392922
121,2025-02-28,1832.696562,1815.079412,1851.177319
122,2025-03-31,1847.027733,1829.511656,1866.974236
123,2025-04-30,1866.787131,1846.731241,1885.166009
124,2025-05-31,1874.536675,1856.988928,1893.085120
125,2025-06-30,1883.439654,1865.153371,1901.967574
126,2025-07-31,1892.294027,1873.660187,1911.230276
127,2025-08-31,1896.911332,1877.847144,1916.442693
128,2025-09-30,1905.019240,1885.328326,1923.972061
129,2025-10-31,1912.996144,1894.407384,1933.029321


## Model 2: Model with Unemployment Regressor


In [5]:
# Technically the same as the combined data set, but creating a copy for cleaner handling
rental_and_employment = combined_data[['ds', 'y', 'employment_count']].copy()
# Limit to end of 2024
rental_and_employment = rental_and_employment[rental_and_employment['ds'] <= '2024-12-31']

m2 = Prophet(holidays=covid)
# Registering the employment data as an additional regressor
m2.add_regressor('employment_count')
m2.fit(rental_and_employment)

# Build the model excluding the year of 2025 
future2 = m2.make_future_dataframe(periods=12, freq='ME')

# Prophet requires that regressor values be present in both fitting and prediction df
# Therefore the employment count will track to 2025 following real data
# First ensure dataframes are of the same type
combined_data['ds'] = pd.to_datetime(combined_data['ds'])
future2['ds'] = pd.to_datetime(future2['ds'])

# Merge data together to add 2025 employment count
future2 = future2.merge(
    combined_data[['ds', 'employment_count']],
    on='ds',
    how='left'
)
# Actual prediction
forecast2 = m2.predict(future2)

# Create a data frame with yhat as the predicted value per row and columns for uncertainty levels
forecast2[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

16:15:24 - cmdstanpy - INFO - Chain [1] start processing
16:15:24 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.ma

,ds,yhat,yhat_lower,yhat_upper
120,2025-01-31,1832.144667,1814.390510,1850.593300
121,2025-02-28,1832.468818,1813.366834,1851.085686
122,2025-03-31,1846.462016,1826.799140,1866.491789
123,2025-04-30,1865.385866,1847.277789,1883.201891
124,2025-05-31,1872.314633,1853.153787,1892.423304
125,2025-06-30,1881.708063,1862.484731,1899.531220
126,2025-07-31,1890.923337,1872.457987,1909.473297
127,2025-08-31,1895.872872,1877.113864,1914.667010
128,2025-09-30,1902.161829,1883.562170,1921.053997
129,2025-10-31,1909.878830,1890.898578,1929.131754


## Midway Comparison
Now that the rental data has been forecasted by the two models, we can do a preliminary comparison before any statistical tests.

In [6]:
# Pull actual values from the year of 2025
actual_values = combined_data[combined_data['ds'] >= '2025-01-01'][['ds', 'y']].copy()

# Pull the model performances 
f1 = forecast1[forecast1['ds'] >= '2025-01-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].rename(
    columns={'yhat': 'yhat_m1', 'yhat_lower': 'lower_m1', 'yhat_upper': 'upper_m1'})
f2 = forecast2[forecast2['ds'] >= '2025-01-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].rename(
    columns={'yhat': 'yhat_m2', 'yhat_lower': 'lower_m2', 'yhat_upper': 'upper_m2'})

# Merge all the data together for easy comparison
midway_comparison = actual_values.merge(f1, on='ds').merge(f2, on='ds')
# Check whether the actual value falls between the upper and lower ends of the model's prediction
midway_comparison['within_m1'] = midway_comparison['y'].between(midway_comparison['lower_m1'], midway_comparison['upper_m1'])
midway_comparison['within_m2'] = midway_comparison['y'].between(midway_comparison['lower_m2'], midway_comparison['upper_m2'])
# Confidence levels
midway_comparison['width_m1'] = midway_comparison['upper_m1'] - midway_comparison['lower_m1']
midway_comparison['width_m2'] = midway_comparison['upper_m2'] - midway_comparison['lower_m2']

print(midway_comparison[['ds', 'y', 'yhat_m1', 'within_m1', 'yhat_m2', 'within_m2']])
print(f"\nM1 coverage: {midway_comparison['within_m1'].mean():.0%}")
print(f"M2 coverage: {midway_comparison['within_m2'].mean():.0%}")
print(f"M1 avg interval width: {midway_comparison['width_m1'].mean():.2f}")
print(f"M2 avg interval width: {midway_comparison['width_m2'].mean():.2f}")

           ds            y      yhat_m1  within_m1      yhat_m2  within_m2
0  2025-01-31  1810.942452  1831.227179      False  1832.144667      False
1  2025-02-28  1853.575750  1832.696562      False  1832.468818      False
2  2025-03-31  1873.731004  1847.027733      False  1846.462016      False
3  2025-04-30  1873.613465  1866.787131       True  1865.385866       True
4  2025-05-31  1885.487821  1874.536675       True  1872.314633       True
5  2025-06-30  1895.184392  1883.439654       True  1881.708063       True
6  2025-07-31  1907.515917  1892.294027       True  1890.923337       True
7  2025-08-31  1898.018927  1896.911332       True  1895.872872       True
8  2025-09-30  1943.949582  1905.019240      False  1902.161829      False
9  2025-10-31  1952.670747  1912.996144      False  1909.878830      False
10 2025-11-30  1947.121214  1915.580941      False  1912.812333      False
11 2025-12-31  1908.264675  1921.073403       True  1919.339272       True

M1 coverage: 50%
M2 cove

## Finetuning 
The models did not perform as well as initially believed, but we will continue with finetuning to see if we can improve the performance.


In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics
import itertools
import numpy as np

# Paraneter grid 
param_grid = {
    'changepoint_prior_scale': [0.01, 0.05, 0.1], 
    'seasonality_prior_scale': [0.1, 1.0, 10.0],
    'changepoint_range': [0.8, 0.9, 0.95]
}

# Generate all combinations of params
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]

# First finetune the base model
rmses_m1 = []  # RMSEs values for the parameters for model 1 

for params in all_params:
    m = Prophet(holidays=covid, **params)
    m.fit(rental_data)
    
    df_cv = cross_validation(m, initial='1825 days', period='180 days', horizon='365 days')
    df_p = performance_metrics(df_cv, rolling_window=1)
    rmses_m1.append(df_p['mae'].values[0])

# Find the best params
best_params_m1 = all_params[np.argmin(rmses_m1)]
print(f" Best parameters for base model: {best_params_m1}")



16:15:25 - cmdstanpy - INFO - Chain [1] start processing
16:15:25 - cmdstanpy - INFO - Chain [1] done processing


  0%|          | 0/8 [00:00<?, ?it/s]

16:15:25 - cmdstanpy - INFO - Chain [1] start processing
16:15:26 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.ma

  0%|          | 0/8 [00:00<?, ?it/s]

16:15:39 - cmdstanpy - INFO - Chain [1] start processing
16:15:40 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.ma

...

16:20:23 - cmdstanpy - INFO - Chain [1] start processing
16:20:24 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.ma

  0%|          | 0/8 [00:00<?, ?it/s]

16:20:38 - cmdstanpy - INFO - Chain [1] start processing
16:20:39 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.ma

  0%|          | 0/8 [00:00<?, ?it/s]

16:20:54 - cmdstanpy - INFO - Chain [1] start processing
16:20:56 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.ma

In [ ]:

# Finetune the second model
rmses_m2 = []  # RMSEs values for the parameters for model 2

for params in all_params:
    m = Prophet(holidays=covid, **params)
    m.add_regressor('employment_count')
    m.fit(rental_and_employment)
    
    df_cv = cross_validation(m, initial='1825 days', period='180 days', horizon='365 days')
    df_p = performance_metrics(df_cv, rolling_window=1)
    rmses_m2.append(df_p['mae'].values[0])

# Find the best params
best_params_m2 = all_params[np.argmin(rmses_m2)]
print(f" Best parameters for second model: {best_params_m2}")

22:56:16 - cmdstanpy - INFO - Chain [1] start processing
22:56:16 - cmdstanpy - INFO - Chain [1] done processing


  0%|          | 0/8 [00:00<?, ?it/s]

22:56:16 - cmdstanpy - INFO - Chain [1] start processing
22:56:17 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.ma

  0%|          | 0/8 [00:00<?, ?it/s]

22:56:30 - cmdstanpy - INFO - Chain [1] start processing
22:56:31 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.ma

...

23:02:34 - cmdstanpy - INFO - Chain [1] start processing
23:02:35 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.ma

  0%|          | 0/8 [00:00<?, ?it/s]

23:02:50 - cmdstanpy - INFO - Chain [1] start processing
23:02:51 - cmdstanpy - INFO - Chain [1] done processing
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.ma

 Best parameters for second model: {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 0.1, 'changepoint_range': 0.95}


/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: invalid value encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1567: RuntimeWarning: divide by zero encountered in matmul
  Xb_m = np.matmul(seasonal_features.values, beta * s_m.values)
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1567: RuntimeWarning: overflow encountered in matmul
  Xb_m = np.matmul(seasonal_features.values, beta * s_m.values)
/Users/jam/.pyenv/versions/MI1-2/li

Now we run fit the models again with their proper hyperparameters

In [ ]:
m1_tuned = Prophet(holidays=covid, **best_params_m1)
m1_tuned.fit(rental_data)

m2_tuned = Prophet(holidays=covid, **best_params_m2)
m2_tuned.add_regressor('employment_count')
m2_tuned.fit(rental_and_employment)

23:07:37 - cmdstanpy - INFO - Chain [1] start processing
23:07:37 - cmdstanpy - INFO - Chain [1] done processing
23:07:38 - cmdstanpy - INFO - Chain [1] start processing
23:07:38 - cmdstanpy - INFO - Chain [1] done processing


And then we can run the forecasts a final time

### Model 1:

In [ ]:
# Build the model excluding the year of 2025 
future = m1_tuned.make_future_dataframe(periods=12, freq='ME')
# Predict 2025 data
forecast1_tuned = m1_tuned.predict(future)

# Create a data frame with yhat as the predicted value per row and columns for uncertainty levels
forecast1_tuned[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.p

,ds,yhat,yhat_lower,yhat_upper
120,2025-01-31,1799.232596,1783.335798,1815.672124
121,2025-02-28,1795.882540,1779.990476,1811.628584
122,2025-03-31,1805.019445,1788.377411,1820.914082
123,2025-04-30,1819.552596,1802.264260,1835.831445
124,2025-05-31,1821.494328,1805.008539,1838.419331
125,2025-06-30,1825.081887,1808.625144,1842.485978
126,2025-07-31,1828.528297,1811.278871,1846.872454
127,2025-08-31,1827.739006,1810.091994,1847.216068
128,2025-09-30,1830.869425,1812.577483,1849.404707
129,2025-10-31,1833.757800,1814.634067,1854.116042


### Model 2

In [ ]:
future2 = m2_tuned.make_future_dataframe(periods=12, freq='ME')

# Prophet requires that regressor values be present in both fitting and prediction df
# Therefore the employment count will track to 2025 following real data
# First ensure dataframes are of the same type
combined_data['ds'] = pd.to_datetime(combined_data['ds'])
future2['ds'] = pd.to_datetime(future2['ds'])

# Merge data together to add 2025 employment count
future2 = future2.merge(
    combined_data[['ds', 'employment_count']],
    on='ds',
    how='left'
)
# Actual prediction
forecast2_tuned = m2_tuned.predict(future2)

# Create a data frame with yhat as the predicted value per row and columns for uncertainty levels
forecast2_tuned[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/jam/.pyenv/versions/MI1-2/lib/python3.10/site-packages/prophet/forecaster.p

,ds,yhat,yhat_lower,yhat_upper
120,2025-01-31,1798.811733,1783.245794,1816.192930
121,2025-02-28,1796.236728,1780.362690,1812.834246
122,2025-03-31,1805.499225,1789.177801,1822.213862
123,2025-04-30,1820.504161,1804.462104,1837.966292
124,2025-05-31,1822.886656,1805.643991,1840.234538
125,2025-06-30,1826.132220,1808.810935,1842.803621
126,2025-07-31,1829.384670,1811.806873,1847.473664
127,2025-08-31,1828.420628,1810.896351,1845.184779
128,2025-09-30,1832.585334,1813.747266,1850.989372
129,2025-10-31,1835.577104,1816.416209,1854.904178


### Final Comparison

In [ ]:
# Pull actual values from the year of 2025
actual_values = combined_data[combined_data['ds'] >= '2025-01-01'][['ds', 'y']].copy()

# Pull the model performances 
f1 = forecast1_tuned[forecast1_tuned['ds'] >= '2025-01-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].rename(
    columns={'yhat': 'yhat_m1', 'yhat_lower': 'lower_m1', 'yhat_upper': 'upper_m1'})
f2 = forecast2_tuned[forecast2_tuned['ds'] >= '2025-01-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].rename(
    columns={'yhat': 'yhat_m2', 'yhat_lower': 'lower_m2', 'yhat_upper': 'upper_m2'})

# Merge all the data together for easy comparison
final_comparison = actual_values.merge(f1, on='ds').merge(f2, on='ds')
# Check whether the actual value falls between the upper and lower ends of the model's prediction
final_comparison['within_m1'] = final_comparison['y'].between(final_comparison['lower_m1'], final_comparison['upper_m1'])
final_comparison['within_m2'] = final_comparison['y'].between(final_comparison['lower_m2'], final_comparison['upper_m2'])
# Confidence levels
final_comparison['width_m1'] = final_comparison['upper_m1'] - final_comparison['lower_m1']
final_comparison['width_m2'] = final_comparison['upper_m2'] - final_comparison['lower_m2']

print(final_comparison[['ds', 'y', 'yhat_m1', 'within_m1', 'yhat_m2', 'within_m2']])
print(f"\nM1 coverage: {final_comparison['within_m1'].mean():.0%}")
print(f"M2 coverage: {final_comparison['within_m2'].mean():.0%}")
print(f"M1 avg interval width: {final_comparison['width_m1'].mean():.2f}")
print(f"M2 avg interval width: {final_comparison['width_m2'].mean():.2f}")

           ds            y      yhat_m1  within_m1      yhat_m2  within_m2
0  2025-01-31  1810.942452  1799.232596       True  1798.811733       True
1  2025-02-28  1853.575750  1795.882540      False  1796.236728      False
2  2025-03-31  1873.731004  1805.019445      False  1805.499225      False
3  2025-04-30  1873.613465  1819.552596      False  1820.504161      False
4  2025-05-31  1885.487821  1821.494328      False  1822.886656      False
5  2025-06-30  1895.184392  1825.081887      False  1826.132220      False
6  2025-07-31  1907.515917  1828.528297      False  1829.384670      False
7  2025-08-31  1898.018927  1827.739006      False  1828.420628      False
8  2025-09-30  1943.949582  1830.869425      False  1832.585334      False
9  2025-10-31  1952.670747  1833.757800      False  1835.577104      False
10 2025-11-30  1947.121214  1831.712779      False  1833.273623      False
11 2025-12-31  1908.264675  1832.380896      False  1833.412242      False

M1 coverage: 8%
M2 cover

After multiple attempts for hyper parameter tuning it seems that the default models performed the best overall.

## Statistical Analysis

Though there was a very small difference between the performance of the two models, we will continue with the statistical test just to prove that there are not different.

In [ ]:
from dieboldmariano import dm_test

# Using the midway values that were not tuned as these were the best
result = dm_test(
    midway_comparison['y'].values,
    midway_comparison['yhat_m1'].values,
    midway_comparison['yhat_m2'].values,
    h=1,          # forecast horizon = 12 months
    one_sided=False # two-sided for H0: M1 == M2
)

dm_stat, p_value = result

print("Diebold-Mariano Test")
print(f"DM Statistic: {dm_stat:.4f}")
print(f"P-value:      {p_value:.4f}")

if p_value < 0.05:
    print("Result: Reject H0. There is a tatistically significant difference between models")
    # The sign of the DM value determines which model performs better. M1 = first model M2 = second
    if dm_stat < 0:
        print("Direction: M2 (employment) outperforms M1 (base)")
    else:
        print("Direction: M1 (base) outperforms M2 (employment)")
else:
    print("Result: Fail to reject H0. There is no statistically significant difference between models")

Diebold-Mariano Test
DM Statistic: -2.6344
P-value:      0.0232
Result: Reject H0 — statistically significant difference between models
Direction: M2 (employment) outperforms M1 (base)


After checking the statistical test, it seems like M2 performed better. However, the DM test focuses on RMSE which penalizes errors. The next cell will look at the mean loss to see how the models were performing. After running the cell it can be seen that M2 has smaller errors overall which explains the results of the DM test.

(Negative values → M2 has smaller error (better)
Positive values → M1 has smaller error)

In [ ]:
# Calculation of the mean loss difference
midway_comparison['e1'] = midway_comparison['y'] - midway_comparison['yhat_m1']
midway_comparison['e2'] = midway_comparison['y'] - midway_comparison['yhat_m2']
midway_comparison['diff'] = midway_comparison['e1']**2 - midway_comparison['e2']**2

print(midway_comparison[['ds', 'diff']])
print("Mean loss diff:", midway_comparison['diff'].mean())

           ds        diff
0  2025-01-31  -38.063773
1  2025-02-28   -9.562073
2  2025-03-31  -30.533035
3  2025-04-30  -21.094555
4  2025-05-31  -53.605286
5  2025-06-30  -43.672575
6  2025-07-31  -43.607800
7  2025-08-31   -3.378783
8  2025-09-30 -230.644806
9  2025-10-31 -257.073972
10 2025-11-30 -182.310442
11 2025-12-31   41.416817
Mean loss diff: -72.67752354253969


Let's check the MAE and perform two more tests to really check what this means for the two models.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Mean absolute error calculation
mae1 = mean_absolute_error(midway_comparison['y'], midway_comparison['yhat_m1'])
mae2 = mean_absolute_error(midway_comparison['y'], midway_comparison['yhat_m2'])

print(f"Model 1 MAE: {mae1}")
print(f"Model 2 MAE: {mae2}")

NameError: name 'midway_comparison' is not defined

In [ ]:
from scipy.stats import ttest_rel
from scipy.stats import wilcoxon

# Absolute error calculation
e1 = abs(midway_comparison['y'] - midway_comparison['yhat_m1'])
e2 = abs(midway_comparison['y'] - midway_comparison['yhat_m2'])

t, p = ttest_rel(e1, e2)
w, p = wilcoxon(e1, e2)

TtestResult(statistic=np.float64(-3.5307270523492664), pvalue=np.float64(0.004708817435788059), df=np.int64(11))
WilcoxonResult(statistic=np.float64(8.0), pvalue=np.float64(0.01220703125))
